# Two Traders with Private Markov-Chain Signals

In this notebook, we compare the learned and exact equilibria of the game in `two_traders.ipynb` with Markov-chain
signals.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import MarkovTraders

## 1. The game

The game is that of `two_traders.ipynb`, with each signal replaced by a two-state Markov chain
$\alpha^i_t \in \{-1, 1\}$ that switches at rate $\kappa = 1$. The signal $\alpha^i$ is private to trader $i$. The
conditional expectation of the other signal is the Wonham filter, which is nonlinear in the observed path. Thus the
equilibrium is not in the linear class of `two_traders.ipynb`.

## 2. Exact equilibrium

At $\lambda = 0$, the trades do not move the price, so the traders decouple. The exact equilibrium combines the
discrete Bayes filter of the other chain with a backward recursion for each trader. At $\lambda = 1$, no exact
equilibrium is known.

## 3. Learned and exact equilibria at $\lambda = 0$

The learned method is described in the README, here with 5,000 paths and damping $\rho = 0.5$. The control variate
also includes the compensated increments of the chains, $\alpha^i_{t_{k+1}} - e^{-2\kappa\Delta t}\,\alpha^i_{t_k}$.
The figure compares the learned and exact controls on three validation paths.

In [ ]:
game = MarkovTraders(lam=0.0)
solver = Solver(game, damping=0.5, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(8, average_from=3, reference=exact)
print(f'relative L2 error: trader 1 {relative_l2(learned[0], exact[0]):.3f}, trader 2 {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['trader 1', 'trader 2'], ylabel=r'$\nu_t$')

## 4. Learned equilibrium at $\lambda = 1$

Without an exact equilibrium, we report the first order condition residual. For each trader, a new network is
fitted to the undamped target on new paths. At an open-loop equilibrium, this network reproduces the control. The
residual is the relative $L^2$ distance between the two on the validation paths.

In [ ]:
game = MarkovTraders(lam=1.0)
solver = Solver(game, damping=0.6, steps=300, batch=1000, n_paths=5000)
learned, _ = solver.run(6, average_from=3)
print('first order condition residuals:', [round(first_order_residual(solver, i), 3) for i in range(2)])
print('mean |nu|:', [round(float(c.abs().mean()), 4) for c in learned])